In [ ]:
# In[1]:

import pandas as pd

# Load CSVs and parse timestamps as UTC datetimes
df_metric = pd.read_csv("metric.csv")
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

df_trace = pd.read_csv("trace.csv")
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

df_log = pd.read_csv("log.csv")
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

df_error = pd.read_csv("error_logs.csv")
df_error['timestamp'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)

# 1) metric.csv: aggregate by cmdb_id and kpi_name (global stats)
metric_grp = df_metric.groupby(['cmdb_id', 'kpi_name']).agg(
    count=('value', 'count'),
    mean=('value', 'mean'),
    p50=('value', lambda x: x.quantile(0.5)),
    p95=('value', lambda x: x.quantile(0.95)),
    max=('value', 'max')
).reset_index()

metric_series_count = metric_grp.shape[0]
metric_top20 = metric_grp.sort_values('p95', ascending=False).head(20).loc[
    :, ['cmdb_id', 'kpi_name', 'count', 'mean', 'p50', 'p95', 'max']
]

# 2) trace.csv: unique trace_name values with counts and top 20 (trace_name, cmdb_id, count)
trace_name_counts = df_trace.groupby('trace_name').size().reset_index(name='count').sort_values('count', ascending=False)
trace_top20 = df_trace.groupby(['trace_name', 'cmdb_id']).size().reset_index(name='count').sort_values('count', ascending=False).head(20)

# 3) log.csv: unique log_name values with counts and top 20 (log_name, cmdb_id, count)
log_name_counts = df_log.groupby('log_name').size().reset_index(name='count').sort_values('count', ascending=False)
log_top20 = df_log.groupby(['log_name', 'cmdb_id']).size().reset_index(name='count').sort_values('count', ascending=False).head(20)

# 4) error_logs.csv: unique cmdb_id counts and up to 5 sample rows
error_unique_cmdb_count = df_error['cmdb_id'].nunique()
error_sample = df_error.loc[:, ['timestamp', 'cmdb_id', 'message']].head(5)

# Compact outputs (limited rows)
metric_series_count, metric_top20, trace_name_counts.head(20), trace_top20, log_name_counts.head(20), log_top20, error_unique_cmdb_count, error_sample

```
Out[1]:
```
summary = (
    "Summary of execution results:\n"
    "- Metrics: 72 unique component-KPI series. Top p95 values are memory KPIs (mem) — "
    "recommendationservice, adservice, currencyservice, emailservice, cartservice, etc. "
    "Other notable high KPIs include adservice diskio and some socket/workload spikes for frontend and recommendationservice.\n"
    "- Traces: most common trace names are trace.from_root.* (171 rows) and trace.from_frontendservice.* (124 rows). "
    "Top trace-series combos include trace.from_frontendservice.duration_mean on currencyservice, frontendservice, and productcatalogservice (25 each).\n"
    "- Logs: two log types (log.error_count and log.row_count) with 227 rows each overall. "
    "log.error_count appears across many services (adservice, cartservice, currencyservice, frontend, recommendationservice, …).\n"
    "- Error logs: only one cmdb_id (frontend) produced errors (count = 1 unique cmdb_id). Sample messages show repeated: "
    "'failed to get product recommendations' at 2024-01-18 13:12:21–13:12:22 UTC.\n\n"
    "Straightforward conclusion:\n"
    "Frontend is repeatedly failing to get product recommendations. The recommendationservice shows the highest memory p95 and also elevated socket/workload metrics, making it the primary suspect. "
    "Recommended next steps: investigate recommendationservice memory usage and socket errors, examine its logs/GC/heap and connections from frontend, and also review adservice/currencyservice memory and adservice disk I/O as secondary checks."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(72,                   cmdb_id  kpi_name  count          mean           p50           p95           max
59  recommendationservice       mem     25  8.927543e+07  4.410109e+07  1.810742e+08  2.044885e+08
4               adservice       mem     25  1.025493e+08  1.024760e+08  1.045044e+08  1.045651e+08
23        currencyservice       mem     25  4.430299e+07  4.428547e+07  4.481040e+07  4.494834e+07
30           emailservice       mem     25  4.341226e+07  4.341555e+07  4.345505e+07  4.346921e+07
10            cartservice       mem     25  4.282272e+07  4.317696e+07  4.343869e+07  4.373579e+07
44         paymentservice       mem     25  2.940830e+07  2.937173e+07  3.263253e+07  3.276418e+07
52  productcatalogservice       mem     25  2.416756e+07  2.420743e+07  2.448906e+07  2.455552e+07
37               frontend       mem     25  1.548448e+07  1.478786e+07  1.710866e+07  1.759027e+07
16        checkoutservice       mem     25  1.073793e+07  1.077712e+07  1.099354e+07  1.111743e+07
69        shippingservice       mem     25  8.334036e+06  8.373794e+06  8.492701e+06  8.559821e+06
64                  redis       mem     25  3.013064e+06  3.001549e+06  3.054619e+06  3.057118e+06
1               adservice    diskio     25  7.207314e+04  7.052779e+04  1.142307e+05  1.295251e+05
63                  redis    diskio     25  2.793760e+03  0.000000e+00  1.368120e+04  1.467544e+04
54  productcatalogservice  workload     25  1.253401e+02  1.373167e+02  1.480683e+02  1.505634e+02
25        currencyservice  workload     25  7.583978e+01  7.689997e+01  8.472779e+01  8.666700e+01
38               frontend    socket     25  3.600333e+01  2.220000e+01  5.800000e+01  5.800000e+01
19        currencyservice       cpu     25  2.862654e+01  2.937829e+01  3.293811e+01  3.382964e+01
12            cartservice  workload     25  2.380704e+01  2.486900e+01  2.612750e+01  2.633300e+01
60  recommendationservice    socket     25  1.043467e+01  5.000000e+00  2.309333e+01  2.593333e+01
61  recommendationservice  workload     25  1.389707e+01  1.800220e+01  1.930321e+01  1.955208e+01,                                       trace_name  count
15                     trace.from_root.row_count    171
14                    trace.from_root.error_rate    171
13                  trace.from_root.duration_p95    171
12                 trace.from_root.duration_mean    171
7           trace.from_frontendservice.row_count    124
6          trace.from_frontendservice.error_rate    124
5        trace.from_frontendservice.duration_p95    124
4       trace.from_frontendservice.duration_mean    124
3           trace.from_checkoutservice.row_count    120
2          trace.from_checkoutservice.error_rate    120
1        trace.from_checkoutservice.duration_p95    120
0       trace.from_checkoutservice.duration_mean    120
36  trace.to_productcatalogservice.duration_mean     98
37   trace.to_productcatalogservice.duration_p95     98
38     trace.to_productcatalogservice.error_rate     98
39      trace.to_productcatalogservice.row_count     98
40  trace.to_recommendationservice.duration_mean     75
41   trace.to_recommendationservice.duration_p95     75
42     trace.to_recommendationservice.error_rate     75
43      trace.to_recommendationservice.row_count     75,                                         trace_name                cmdb_id  count
21        trace.from_frontendservice.duration_mean        currencyservice     25
22        trace.from_frontendservice.duration_mean        frontendservice     25
23        trace.from_frontendservice.duration_mean  productcatalogservice     25
44     trace.from_recommendationservice.error_rate  productcatalogservice     25
43   trace.from_recommendationservice.duration_p95  recommendationservice     25
54                   trace.from_root.duration_mean  recommendationservice     25
50                   trace.from_root.duration_mean           emailservice     25
51                   trace.from_root.duration_mean        frontendservice     25
40  trace.from_recommendationservice.duration_mean  productcatalogservice     25
41  trace.from_recommendationservice.duration_mean  recommendationservice     25
42   trace.from_recommendationservice.duration_p95  productcatalogservice     25
39            trace.from_frontendservice.row_count  recommendationservice     25
36            trace.from_frontendservice.row_count        currencyservice     25
34           trace.from_frontendservice.error_rate  recommendationservice     25
37            trace.from_frontendservice.row_count        frontendservice     25
38            trace.from_frontendservice.row_count  productcatalogservice     25
32           trace.from_frontendservice.error_rate        frontendservice     25
33           trace.from_frontendservice.error_rate  productcatalogservice     25
29         trace.from_frontendservice.duration_p95  recommendationservice     25
31           trace.from_frontendservice.error_rate        currencyservice     25,           log_name  count
0  log.error_count    227
1    log.row_count    227,            log_name                cmdb_id  count
0   log.error_count              adservice     25
1   log.error_count            cartservice     25
3   log.error_count        currencyservice     25
5   log.error_count               frontend     25
17    log.row_count  recommendationservice     25
7   log.error_count  recommendationservice     25
10    log.row_count              adservice     25
9   log.error_count        shippingservice     25
11    log.row_count            cartservice     25
19    log.row_count        shippingservice     25
15    log.row_count               frontend     25
13    log.row_count        currencyservice     25
2   log.error_count        checkoutservice     24
14    log.row_count           emailservice     24
4   log.error_count           emailservice     24
6   log.error_count         paymentservice     24
16    log.row_count         paymentservice     24
12    log.row_count        checkoutservice     24
8   log.error_count                  redis      5
18    log.row_count                  redis      5, 1,                   timestamp   cmdb_id                                message
0 2024-01-18 13:12:21+00:00  frontend  failed to get product recommendations
1 2024-01-18 13:12:21+00:00  frontend  failed to get product recommendations
2 2024-01-18 13:12:22+00:00  frontend  failed to get product recommendations
3 2024-01-18 13:12:22+00:00  frontend  failed to get product recommendations
4 2024-01-18 13:12:22+00:00  frontend  failed to get product recommendations)```
```

In [ ]:
# In[2]:

import pandas as pd
# Reuse previously loaded dataframes: df_metric, df_trace, df_log, df_error
# Ensure timestamps are parsed (they were parsed previously, but enforce to be safe)
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)
df_error['timestamp'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)

# Allowed root-cause components
allowed = ["checkoutservice","currencyservice","emailservice","productcatalogservice","recommendationservice"]

# Incident window (UTC)
start = pd.to_datetime("2024-01-18 12:55:59", utc=True)
end = pd.to_datetime("2024-01-18 13:25:59", utc=True)

# 1) Metrics (primary) - compute global P95 before filtering to window
metric_allowed = df_metric[df_metric['cmdb_id'].isin(allowed)].copy()

# Global p95 per series (cmdb_id, kpi_name)
global_p95 = (
    metric_allowed
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .quantile(0.95)
    .reset_index(name='global_p95')
)

# Filter metric to incident window (and allowed components)
metric_window = metric_allowed[(metric_allowed['timestamp'] >= start) & (metric_allowed['timestamp'] <= end)].copy()

# Join windowed points with global p95 and find anomalies where value > global_p95
metric_win_with_p95 = metric_window.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')

anomalous_points = metric_win_with_p95[metric_win_with_p95['value'] > metric_win_with_p95['global_p95']].copy()

if anomalous_points.empty:
    metric_anomalies_summary = "no metric anomalies found"
else:
    metric_anomalies = (
        anomalous_points
        .groupby(['cmdb_id', 'kpi_name', 'global_p95'], as_index=False)
        .agg(
            earliest_anomaly_timestamp=('timestamp', 'min'),
            anomaly_count_in_window=('value', 'count'),
            max_value_in_window=('value', 'max')
        )
    )
    metric_anomalies['severity_ratio'] = metric_anomalies['max_value_in_window'] / metric_anomalies['global_p95']
    # keep compact, sort by severity_ratio desc and limit 20
    metric_anomalies_summary = (
        metric_anomalies
        .sort_values('severity_ratio', ascending=False)
        .loc[:, ['cmdb_id', 'kpi_name', 'global_p95', 'earliest_anomaly_timestamp', 'anomaly_count_in_window', 'max_value_in_window', 'severity_ratio']]
        .head(20)
        .reset_index(drop=True)
    )

# 2) Traces (secondary) - filter to window and select recommendationservice/productcatalogservice references
trace_window = df_trace[(df_trace['timestamp'] >= start) & (df_trace['timestamp'] <= end)].copy()
trace_filter_mask = (
    trace_window['trace_name'].str.contains('recommendationservice|productcatalogservice', na=False)
    | trace_window['cmdb_id'].isin(['recommendationservice','productcatalogservice'])
)
trace_selected = trace_window[trace_filter_mask].copy()

if trace_selected.empty:
    trace_agg_summary = pd.DataFrame(columns=['trace_name','cmdb_id','count','mean_value','p95_value','max_value'])
else:
    trace_agg = (
        trace_selected
        .groupby(['trace_name', 'cmdb_id'])['value']
        .agg(count='count', mean_value='mean', p95_value=lambda x: x.quantile(0.95), max_value='max')
        .reset_index()
    )
    trace_agg_summary = trace_agg.sort_values('mean_value', ascending=False).head(20).reset_index(drop=True)

# 3) Logs and error_logs (tertiary)
log_window = df_log[(df_log['timestamp'] >= start) & (df_log['timestamp'] <= end)].copy()
if log_window.empty:
    logs_agg_summary = pd.DataFrame(columns=['cmdb_id','log_name','count','sum_value'])
else:
    logs_agg = (
        log_window
        .groupby(['cmdb_id', 'log_name'])['value']
        .agg(count='count', sum_value='sum')
        .reset_index()
        .sort_values('count', ascending=False)
    )
    logs_agg_summary = logs_agg.head(20).reset_index(drop=True)

error_window = df_error[(df_error['timestamp'] >= start) & (df_error['timestamp'] <= end)].copy()
if error_window.empty:
    error_counts = pd.DataFrame(columns=['cmdb_id','count'])
    error_samples = pd.DataFrame(columns=['timestamp','cmdb_id','message'])
else:
    error_counts = error_window.groupby('cmdb_id').size().reset_index(name='count').sort_values('count', ascending=False).reset_index(drop=True)
    error_samples = error_window.loc[:, ['timestamp','cmdb_id','message']].head(10).reset_index(drop=True)

# Final compact outputs: metric_anomalies_summary (or message), trace_agg_summary, logs_agg_summary, error_counts, error_samples
metric_anomalies_summary, trace_agg_summary, logs_agg_summary, error_counts, error_samples

```
Out[2]:
```
summary = (
    "Findings (concise):\n"
    "- Metric anomalies (P95-based) detected for the allowed services. The strongest signals point to recommendationservice: multiple KPIs exceeded their global P95 in the incident window (latency-50/90, cpu, mem, socket), with severity ratios up to ~1.38 for latency and ~1.13 for memory. productcatalogservice and checkoutservice show smaller but notable latency and error spikes.\n"
    "- Trace aggregation in the window shows high row_counts and elevated duration p95 for traces involving productcatalogservice and recommendationservice (recommendationservice duration_p95 ~1.70s).\n"
    "- Logs and error_logs: frontend produced many errors in the window (865 error_count) and all sampled error messages are \"failed to get product recommendations\". recommendationservice logs do not show error_count in the window.\n\n"
    "Straightforward conclusion:\n"
    "Frontend failures to obtain product recommendations are symptomatic; telemetry implicates recommendationservice as the primary root cause (high latencies, CPU, memory and socket metrics exceeding global P95). productcatalogservice shows secondary latency anomalies and should be checked as a dependent service.\n\n"
    "Recommended next actions:\n"
    "1) Investigate recommendationservice immediately: check heap/memory usage, GC, CPU, socket/connection limits, and threadpool or resource exhaustion. 2) Inspect traces between frontend → recommendationservice to correlate slow requests and error paths. 3) If needed, restart or scale recommendationservice, and monitor frontend errors and recommendationservice metrics. 4) Also review productcatalogservice latency as a secondary check."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name    global_p95 earliest_anomaly_timestamp  anomaly_count_in_window  max_value_in_window  severity_ratio
0         checkoutservice  latency-90  2.973766e-01  2024-01-18 13:13:00+00:00                        2         5.486922e-01        1.845109
1   recommendationservice  latency-50  5.884589e-01  2024-01-18 13:14:00+00:00                        2         8.127657e-01        1.381177
2   recommendationservice  latency-90  7.634131e+00  2024-01-18 13:16:00+00:00                        2         1.044710e+01        1.368472
3   productcatalogservice  latency-90  4.927917e-03  2024-01-18 13:12:00+00:00                        2         6.550811e-03        1.329327
4   recommendationservice         cpu  3.138193e+00  2024-01-18 13:14:00+00:00                        2         3.659352e+00        1.166070
5   recommendationservice         mem  1.810742e+08  2024-01-18 13:18:00+00:00                        2         2.044885e+08        1.129307
6   recommendationservice      socket  2.309333e+01  2024-01-18 13:19:00+00:00                        2         2.593333e+01        1.122979
7   productcatalogservice       error  1.533500e-01  2024-01-18 13:16:00+00:00                        1         1.667500e-01        1.087382
8         checkoutservice  latency-50  1.258080e-01  2024-01-18 13:14:00+00:00                        2         1.338362e-01        1.063814
9   productcatalogservice  latency-50  2.818215e-03  2024-01-18 13:14:00+00:00                        2         2.983582e-03        1.058678
10        checkoutservice      socket  1.000000e+01  2024-01-18 13:14:00+00:00                        1         1.056667e+01        1.056667
11  productcatalogservice         cpu  4.248905e+00  2024-01-18 12:59:00+00:00                        2         4.445045e+00        1.046163
12        currencyservice  latency-90  3.859732e-02  2024-01-18 13:06:00+00:00                        2         4.016685e-02        1.040664
13           emailservice    workload  1.067600e+00  2024-01-18 13:05:00+00:00                        2         1.107700e+00        1.037561
14        currencyservice         cpu  3.293811e+01  2024-01-18 12:58:00+00:00                        2         3.382964e+01        1.027067
15        checkoutservice    workload  1.062703e+00  2024-01-18 12:58:00+00:00                        2         1.091133e+00        1.026753
16           emailservice         cpu  3.393734e-01  2024-01-18 12:58:00+00:00                        2         3.483640e-01        1.026492
17        checkoutservice         cpu  4.921442e-01  2024-01-18 12:58:00+00:00                        2         5.041858e-01        1.024468
18        currencyservice    workload  8.472779e+01  2024-01-18 12:58:00+00:00                        2         8.666700e+01        1.022888
19           emailservice  latency-90  5.014845e-03  2024-01-18 13:14:00+00:00                        2         5.108718e-03        1.018719,                                        trace_name                cmdb_id  count   mean_value    p95_value    max_value
0            trace.from_frontendservice.row_count  productcatalogservice     25  3099.320000  3806.600000  3835.000000
1        trace.to_productcatalogservice.row_count        frontendservice     25  3099.320000  3806.600000  3835.000000
2      trace.from_recommendationservice.row_count  recommendationservice     25   462.520000   583.400000   591.000000
3            trace.from_frontendservice.row_count  recommendationservice     25   462.520000   583.400000   591.000000
4        trace.to_recommendationservice.row_count  recommendationservice     25   462.520000   583.400000   591.000000
5        trace.to_recommendationservice.row_count        frontendservice     25   462.520000   583.400000   591.000000
6        trace.to_productcatalogservice.row_count  recommendationservice     25   460.560000   583.200000   591.000000
7      trace.from_recommendationservice.row_count  productcatalogservice     25   460.560000   583.200000   591.000000
8            trace.from_checkoutservice.row_count  productcatalogservice     24    63.291667    85.800000    90.000000
9        trace.to_productcatalogservice.row_count        checkoutservice     24    63.291667    85.800000    90.000000
10                      trace.from_root.row_count  productcatalogservice     24    31.500000    61.150000   118.000000
11       trace.to_productcatalogservice.row_count                   root     24    31.500000    61.150000   118.000000
12                      trace.from_root.row_count  recommendationservice     25    17.800000    24.000000    24.000000
13       trace.to_recommendationservice.row_count                   root     25    17.800000    24.000000    24.000000
14    trace.to_recommendationservice.duration_p95        frontendservice     25     0.602790     1.701812     1.793006
15        trace.from_frontendservice.duration_p95  recommendationservice     25     0.602790     1.701812     1.793006
16  trace.from_recommendationservice.duration_p95  recommendationservice     25     0.601862     1.700975     1.792384
17    trace.to_recommendationservice.duration_p95  recommendationservice     25     0.601862     1.700975     1.792384
18   trace.to_recommendationservice.duration_mean        frontendservice     25     0.138039     0.364480     0.550744
19       trace.from_frontendservice.duration_mean  recommendationservice     25     0.138039     0.364480     0.550744,                   cmdb_id         log_name  count  sum_value
0               adservice  log.error_count     25          0
1               adservice    log.row_count     25      10233
2             cartservice  log.error_count     25          0
3             cartservice    log.row_count     25      17333
4         currencyservice    log.row_count     25      54206
5         currencyservice  log.error_count     25          0
6                frontend  log.error_count     25        865
7                frontend    log.row_count     25      61271
8         shippingservice    log.row_count     25       9916
9         shippingservice  log.error_count     25          0
10  recommendationservice  log.error_count     25          0
11  recommendationservice    log.row_count     25      11702
12           emailservice  log.error_count     24          0
13           emailservice    log.row_count     24        615
14        checkoutservice    log.row_count     24       1845
15        checkoutservice  log.error_count     24          0
16         paymentservice  log.error_count     24          0
17         paymentservice    log.row_count     24       1230
18                  redis    log.row_count      5         25
19                  redis  log.error_count      5          0,     cmdb_id  count
0  frontend    865,                   timestamp   cmdb_id                                message
0 2024-01-18 13:12:21+00:00  frontend  failed to get product recommendations
1 2024-01-18 13:12:21+00:00  frontend  failed to get product recommendations
2 2024-01-18 13:12:22+00:00  frontend  failed to get product recommendations
3 2024-01-18 13:12:22+00:00  frontend  failed to get product recommendations
4 2024-01-18 13:12:22+00:00  frontend  failed to get product recommendations
5 2024-01-18 13:12:22+00:00  frontend  failed to get product recommendations
6 2024-01-18 13:12:22+00:00  frontend  failed to get product recommendations
7 2024-01-18 13:12:22+00:00  frontend  failed to get product recommendations
8 2024-01-18 13:12:22+00:00  frontend  failed to get product recommendations
9 2024-01-18 13:12:22+00:00  frontend  failed to get product recommendations)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse existing df_metric; ensure timestamp is UTC datetime (idempotent)
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], utc=True)

# Allowed components and incident window
allowed = ["checkoutservice","currencyservice","emailservice","productcatalogservice","recommendationservice"]
start = pd.to_datetime("2024-01-18 12:55:59", utc=True)
end = pd.to_datetime("2024-01-18 13:25:59", utc=True)

# 1) Compute global P95 per (cmdb_id, kpi_name) for allowed components using full series
metric_allowed_full = df_metric[df_metric['cmdb_id'].isin(allowed)].copy()
global_p95 = (
    metric_allowed_full
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .quantile(0.95)
    .reset_index(name='global_p95')
)

# 2) Filter metric rows to the incident window for allowed components
metric_window = metric_allowed_full[(metric_allowed_full['timestamp'] >= start) & (metric_allowed_full['timestamp'] <= end)].copy()

# Prepare list to collect fault segments
segments = []

# Iterate per component and kpi_name
for (cmdb, kpi), grp_global in global_p95.groupby(['cmdb_id', 'kpi_name']):
    threshold = grp_global['global_p95'].iloc[0]
    # select window rows for this series
    s = metric_window[(metric_window['cmdb_id'] == cmdb) & (metric_window['kpi_name'] == kpi)].copy()
    if s.empty:
        continue
    # mark anomaly points where value > global_p95
    s = s.sort_values('timestamp')
    s['is_fault'] = s['value'] > threshold
    fault_points = s[s['is_fault']].copy()
    if fault_points.empty:
        continue
    # group consecutive timestamps into segments (consecutive minutes)
    fault_points['ts_diff'] = fault_points['timestamp'].diff()
    # start new segment when diff != 1 minute or NaT (first row)
    fault_points['new_segment'] = (fault_points['ts_diff'] != pd.Timedelta(minutes=1)).fillna(True)
    fault_points['segment_id'] = fault_points['new_segment'].cumsum()
    # aggregate per segment
    seg_agg = (
        fault_points
        .groupby('segment_id', as_index=False)
        .agg(
            cmdb_id=('cmdb_id', 'first'),
            kpi_name=('kpi_name', 'first'),
            global_p95=('value', lambda x, thr=threshold: threshold),  # will set later
            fault_start_timestamp=('timestamp', 'min'),
            fault_end_timestamp=('timestamp', 'max'),
            point_count_in_fault=('value', 'count'),
            max_value_in_fault=('value', 'max')
        )
    )
    # set global_p95 properly and severity_ratio
    seg_agg['global_p95'] = threshold
    seg_agg['severity_ratio'] = np.where(seg_agg['global_p95'] > 0, seg_agg['max_value_in_fault'] / seg_agg['global_p95'], np.nan)
    # collect
    segments.append(seg_agg.loc[:, ['cmdb_id','kpi_name','global_p95','fault_start_timestamp','fault_end_timestamp','point_count_in_fault','max_value_in_fault','severity_ratio']])

# concatenate all segments
if segments:
    segment_df = pd.concat(segments, ignore_index=True)
    # sort by fault_start asc then severity_ratio desc
    segment_df = segment_df.sort_values(['fault_start_timestamp','severity_ratio'], ascending=[True, False]).reset_index(drop=True)
    segment_table = segment_df.head(20).copy()
else:
    # empty structure with required columns
    segment_table = pd.DataFrame(columns=['cmdb_id','kpi_name','global_p95','fault_start_timestamp','fault_end_timestamp','point_count_in_fault','max_value_in_fault','severity_ratio'])

# 4B) Per-component summary (one row per allowed service)
summary_rows = []
for cmdb in allowed:
    comp_segs = segment_df[segment_df['cmdb_id'] == cmdb] if not segment_df.empty else pd.DataFrame()
    total_fault_segments = 0 if comp_segs.empty else comp_segs.shape[0]
    earliest = pd.NaT if comp_segs.empty else comp_segs['fault_start_timestamp'].min()
    latest = pd.NaT if comp_segs.empty else comp_segs['fault_end_timestamp'].max()
    max_sev = np.nan if comp_segs.empty else comp_segs['severity_ratio'].max()
    summary_rows.append({
        'cmdb_id': cmdb,
        'total_fault_segments_in_window': total_fault_segments,
        'earliest_fault_start_timestamp': earliest,
        'latest_fault_end_timestamp': latest,
        'max_severity_ratio_across_segments': max_sev
    })

per_component_summary = pd.DataFrame(summary_rows)

# Keep outputs compact and bounded
segment_table, per_component_summary

```
Out[4]:
```
Summary of findings (plain English):

- Overall: All five allowed services showed one or more fault segments (value > global P95) inside the incident window, but most exceedances are modest (severity ratios close to 1). A few segments stand out with larger exceedance.

Per-service highlights (one-line each):
- checkoutservice — 9 fault segments; earliest 2024-01-18 12:58 UTC, latest 2024-01-18 13:21 UTC; max severity_ratio 1.845 (most severe single segment: latency-90 exceeded global P95 by ~1.85x).
- currencyservice — 10 fault segments; earliest 2024-01-18 12:58 UTC, latest 2024-01-18 13:22 UTC; max severity_ratio 1.041 (small exceedances, CPU/latency/workload).
- emailservice — 9 fault segments; earliest 2024-01-18 12:58 UTC, latest 2024-01-18 13:22 UTC; max severity_ratio 1.038 (small exceedances, CPU/workload).
- productcatalogservice — 10 fault segments; earliest 2024-01-18 12:58 UTC, latest 2024-01-18 13:18 UTC; max severity_ratio 1.329 (notable latency/error segments and small mem/cpu blips).
- recommendationservice — 11 fault segments; earliest 2024-01-18 12:59 UTC, latest 2024-01-18 13:20 UTC; max severity_ratio 1.381 (multiple KPIs: latency, cpu, mem, socket exceeded P95).

Notable per-segment observations (earliest segments, ordered by time):
- 2024-01-18 12:58 UTC: currencyservice CPU exceeded P95 (ratio ~1.027).
- 2024-01-18 12:58–12:59 UTC: emailservice and checkoutservice had CPU and workload segments (ratios ~1.026–1.037).
- 2024-01-18 13:12–13:18 UTC: productcatalogservice had latency-90 and mem small exceedances (latency ratio ~1.33).
- 2024-01-18 13:14–13:19 UTC: recommendationservice had multiple KPI exceedances including latency (ratio up to ~1.38), CPU, mem, socket.

Interpretation and recommended next steps:
- The single largest spike by severity is checkoutservice latency-90 (severity ~1.85x). Recommendationservice has multiple elevated KPIs across latency, CPU, memory and sockets and is the most consistent multi-KPI offender.
- Prioritize investigation of recommendationservice (memory/CPU/socket/latency) because it shows repeated KPI exceedances across types. Also investigate the checkoutservice latency spike (very high single-segment severity).
- Check traces, logs, thread/connection counts and resource usage for recommendationservice and checkoutservice; correlate with frontend errors/requests to confirm impact path. Productcatalogservice is a secondary candidate to inspect for latency-related impacts.

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name    global_p95     fault_start_timestamp       fault_end_timestamp  point_count_in_fault  max_value_in_fault  severity_ratio
0         currencyservice         cpu  3.293811e+01 2024-01-18 12:58:00+00:00 2024-01-18 12:58:00+00:00                     1        3.382964e+01        1.027067
1            emailservice         cpu  3.393734e-01 2024-01-18 12:58:00+00:00 2024-01-18 12:59:00+00:00                     2        3.483640e-01        1.026492
2         checkoutservice         cpu  4.921442e-01 2024-01-18 12:58:00+00:00 2024-01-18 12:59:00+00:00                     2        5.041858e-01        1.024468
3         currencyservice    workload  8.472779e+01 2024-01-18 12:58:00+00:00 2024-01-18 12:58:00+00:00                     1        8.666700e+01        1.022888
4   productcatalogservice    workload  1.480683e+02 2024-01-18 12:58:00+00:00 2024-01-18 12:59:00+00:00                     2        1.505634e+02        1.016851
5         checkoutservice    workload  1.062703e+00 2024-01-18 12:58:00+00:00 2024-01-18 12:58:00+00:00                     1        1.067000e+00        1.004043
6   productcatalogservice         cpu  4.248905e+00 2024-01-18 12:59:00+00:00 2024-01-18 12:59:00+00:00                     1        4.445045e+00        1.046163
7   recommendationservice    workload  1.930321e+01 2024-01-18 12:59:00+00:00 2024-01-18 12:59:00+00:00                     1        1.955208e+01        1.012893
8   productcatalogservice         mem  2.448906e+07 2024-01-18 13:01:00+00:00 2024-01-18 13:01:00+00:00                     1        2.455552e+07        1.002714
9   productcatalogservice         cpu  4.248905e+00 2024-01-18 13:01:00+00:00 2024-01-18 13:01:00+00:00                     1        4.254347e+00        1.001281
10        currencyservice         cpu  3.293811e+01 2024-01-18 13:02:00+00:00 2024-01-18 13:02:00+00:00                     1        3.306206e+01        1.003763
11        currencyservice         mem  4.481040e+07 2024-01-18 13:03:00+00:00 2024-01-18 13:03:00+00:00                     1        4.494834e+07        1.003078
12           emailservice    workload  1.067600e+00 2024-01-18 13:05:00+00:00 2024-01-18 13:05:00+00:00                     1        1.107700e+00        1.037561
13        checkoutservice    workload  1.062703e+00 2024-01-18 13:05:00+00:00 2024-01-18 13:05:00+00:00                     1        1.091133e+00        1.026753
14        currencyservice  latency-90  3.859732e-02 2024-01-18 13:06:00+00:00 2024-01-18 13:06:00+00:00                     1        4.016685e-02        1.040664
15        currencyservice    workload  8.472779e+01 2024-01-18 13:06:00+00:00 2024-01-18 13:06:00+00:00                     1        8.502777e+01        1.003540
16           emailservice    workload  1.067600e+00 2024-01-18 13:07:00+00:00 2024-01-18 13:07:00+00:00                     1        1.067750e+00        1.000141
17  recommendationservice    workload  1.930321e+01 2024-01-18 13:08:00+00:00 2024-01-18 13:08:00+00:00                     1        1.936227e+01        1.003059
18  productcatalogservice         mem  2.448906e+07 2024-01-18 13:11:00+00:00 2024-01-18 13:11:00+00:00                     1        2.452145e+07        1.001323
19  productcatalogservice  latency-90  4.927917e-03 2024-01-18 13:12:00+00:00 2024-01-18 13:12:00+00:00                     1        4.933719e-03        1.001177,                  cmdb_id  total_fault_segments_in_window earliest_fault_start_timestamp latest_fault_end_timestamp  max_severity_ratio_across_segments
0        checkoutservice                               9      2024-01-18 12:58:00+00:00  2024-01-18 13:21:00+00:00                            1.845109
1        currencyservice                              10      2024-01-18 12:58:00+00:00  2024-01-18 13:22:00+00:00                            1.040664
2           emailservice                               9      2024-01-18 12:58:00+00:00  2024-01-18 13:22:00+00:00                            1.037561
3  productcatalogservice                              10      2024-01-18 12:58:00+00:00  2024-01-18 13:18:00+00:00                            1.329327
4  recommendationservice                              11      2024-01-18 12:59:00+00:00  2024-01-18 13:20:00+00:00                            1.381177)```
```